# Trend Scout — мультиагентний research-дайджест на LangGraph

**Фінальний проєкт** · курс Generative AI Developer (robot_dreams) · Роман Мицько · липень 2026

Репозиторій: https://github.com/RomanMytsko/trend-scout

**Ідея.** Щотижневий моніторинг agentic-новин — рутина з чіткою декомпозицією: знайти → відібрати → написати → перевірити. Trend Scout автоматизує її мультиагентною системою за патерном **orchestrator–workers**:

```
planner ──> researcher ──> curator ──> writer ──> judge ──> END
   ^            │                        ^          │
   └── replan ──┘ (мало матеріалу)       └─ revise ─┘ (оцінка < 4.0)
```

| Вузол | Тип | Відповідальність |
|---|---|---|
| planner | LLM-агент | декомпозиція тем у пошукові запити (structured output) |
| researcher | tool-воркер | RSS + DuckDuckGo news, дедуплікація; детермінований |
| curator | LLM-агент | ранжування/фільтрація під аудиторію, відсів маркетингу |
| writer | LLM-агент | дайджест у строгому форматі; враховує фідбек judge |
| judge | guardrail + LLM-as-a-judge | URL-allowlist перевірка + рубрика 1–5 за 3 критеріями |

Дві петлі зворотного звʼязку (replan і revise) обмежені лічильниками — гарантія завершення.

In [ ]:
%pip install -q git+https://github.com/RomanMytsko/trend-scout.git

## API-ключ

Потрібен лише `OPENAI_API_KEY` (пошук і RSS — безключові). У Colab додай ключ у **Secrets** (іконка 🔑 зліва) або введи нижче.

In [ ]:
import os

if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        import getpass
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

# Мова дайджесту і аудиторія конфігуруються через env (див. .env.example)
os.environ.setdefault("DIGEST_LANGUAGE", "Ukrainian")
os.environ.setdefault("AUDIENCE", "backend Python engineer")
print("OK")

## Візуалізація графа

LangGraph компілює `StateGraph` з умовними переходами — подивимось на нього.

In [ ]:
from trend_scout import graph

app = graph.build_graph()

try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_mermaid())  # fallback: mermaid-текст

## Запуск пайплайну

Теми можна міняти на свої. Виконання ~1–3 хв: RSS-фіди, кілька пошукових запитів і 4–6 LLM-викликів.

In [ ]:
TOPICS = [
    "multi-agent orchestration",
    "MCP and A2A protocols",
    "LangGraph and agent frameworks",
    "context engineering",
]

state = graph.run_digest(TOPICS)

print("--- pipeline events ---")
for event in state["events"]:
    print(" ", event)

## Оцінка якості (LLM-as-a-judge)

Перед рубрикою LLM-судді спрацьовує **детермінований guardrail**: усі лінки в дайджесті мають входити до allowlist зібраних URL (захист від галюцинованих джерел і prompt injection у сніпетах).

In [ ]:
verdict = state["verdict"]
print(f"relevance    : {verdict.relevance}/5")
print(f"grounding    : {verdict.grounding}/5")
print(f"format       : {verdict.format_score}/5")
print(f"average      : {verdict.average:.2f} (поріг 4.0)")
print(f"ревізій      : {state.get('revisions', 0)}")
print(f"replan-ів    : {state.get('replans', 0)}")
print(f"\nфідбек судді: {verdict.feedback}")

## Результат — дайджест

In [ ]:
from IPython.display import Markdown, display

display(Markdown(state["digest"]))

## Зазирнемо всередину: проміжні стани агентів

In [ ]:
print("PLAN (planner, structured output):")
print("  reasoning:", state["plan"].reasoning)
for q in state["plan"].queries:
    print("  query:", q)

print(f"\nITEMS (researcher): {len(state['items'])} зібрано, приклади:")
for item in state["items"][:5]:
    print(f"  [{item.source}] {item.title[:80]}")

print("\nCURATION (curator):")
for pick in state["curation"].picks:
    print(f"  #{pick.index} relevance={pick.relevance}: {pick.why_it_matters[:100]}")

## Висновки

- **Патерн orchestrator–workers** реалізований на LangGraph: LLM-агенти планують/курують/пишуть, детермінований воркер виконує інструменти.
- **Structured outputs (Pydantic)** у кожного агента — жодного парсингу вільного тексту.
- **LLM-as-a-judge** як quality gate з петлею ревізій і порогом 4.0.
- **Guardrails проти prompt injection**: санітизація контенту, явне маркування untrusted-даних у промптах, детермінований URL-allowlist.
- **Надійність** як у бекенд-системах: graceful degradation інструментів, capped retries, event log.

Можливий розвиток: доставка в Telegram, розклад на Celery beat / cron, A2A-інтерфейс для інших агентів.